# Hard-pair 采集（Jupyter 版）

每个场景一个 cell。流程：

1. 先跑 **setup** 加载 overlay / 启动视频流；
2. 反复跑 **preview** 看当前 `f_luma`、四象限 `u`；
3. 摆好场景后跑对应场景 cell，**按回车**才开始采集（想准备多久都行）；
4. 采完自动写 `data/hard_pairs_sceneN.csv` 并合并到 `data/hard_pairs.csv`；
5. 最后跑 merge/eval 和 cleanup。

如果 setup 报 `Root permissions required`，说明当前 Jupyter kernel 不是 root：
用 sudo 启动 Jupyter，或改用终端里的 `tools/collect_hard_pairs.py`。


In [ ]:
import os, sys
ROOT = os.path.abspath(os.getcwd())
while ROOT != os.path.dirname(ROOT) and not os.path.isfile(os.path.join(ROOT, 'lib', 'video_setup.py')):
    ROOT = os.path.dirname(ROOT)
sys.path.insert(0, ROOT)
sys.path.insert(0, os.path.join(ROOT, 'tools'))
import hard_pair_notebook as hp
print('ROOT =', ROOT)
print('euid =', os.geteuid())


In [ ]:
# 摄像头已经由别的 notebook 配好时，把 skip_camera 改成 True
ol, regs, vdma0, vdma1 = hp.setup(skip_camera=False, display=True)


## Preview

摆场景时反复跑这一格。看 `f_luma` 是否在目标带、`u` 是否明显。

* 均匀场景：`u` 应该接近 0；
* 有窗/灯的逆光：`u` 应该明显大于 0（先按 0.3 看）。


In [ ]:
hp.preview(regs)


## Scene 0：均匀暗场

灰卡/白墙/棋盘，目标 `f_luma` 10~30。不要只遮黑镜头。


In [ ]:
rows0 = hp.collect_scene(regs, 0, n=60, wait=10)


## Scene 1：暗场 + 亮窗/灯

房间仍然暗，但让窗/灯进入画面；`f_luma` 仍保持在 10~30，
目标是 `u` 明显变大。


In [ ]:
rows1 = hp.collect_scene(regs, 1, n=60, wait=10)

## Scene 2：均匀正常光

灰卡/白墙/纹理卡，目标 `f_luma` 90~100；不要放窗/灯。


In [ ]:
rows2 = hp.collect_scene(regs, 2, n=60, wait=10)


## Scene 3：正常亮度 + 逆光

窗/灯进入画面，主体背光；`f_luma` 仍保持在 90~100，
目标是 `u` 明显变大、MLP 可能仍判 2，但空间决策应改判 1。


In [ ]:
rows3 = hp.collect_scene(regs, 3, n=60, wait=10)


## 合并 + 评估

把四个场景合并成 `data/hard_pairs.csv`，并打印空间决策表。


In [ ]:
hp.merge_and_print()
hp.evaluate()


## 收尾

停 VDMA、释放 overlay。下次采集前先 Restart Kernel。


In [ ]:
hp.cleanup(ol, vdma0, vdma1)
